# Test end-to-end SK-RD4AD: export ONNX + inference_simulation

Valida l'intera catena su GPU Colab:
1. export del checkpoint in ONNX puro (pesi veri, metadati embedded)
2. verifica parity PyTorch ↔ ONNX
3. inferenza con `inference_simulation` (crop dinamico + blur + score auto-configurati dai metadati)
4. ispezione visiva di heatmap e score

**Prerequisiti**: runtime GPU (Runtime → Cambia tipo di runtime → T4), checkpoint `.pth` e immagini di test su Google Drive.

In [ ]:
# 1) Ambiente: clone dei repo + dipendenze
!nvidia-smi -L
!git clone https://github.com/EmanuelePietroCometti/SK-RD4AD.git
!git clone https://github.com/EmanuelePietroCometti/inference_simulation.git
# torch/torchvision/opencv sono già in Colab; serve solo onnxruntime-gpu (+ onnx).
# NB: onnxruntime-gpu >=1.23 su PyPI è compilato per CUDA 13 (libcudart.so.13),
# ma Colab monta CUDA 12 -> pinniamo la 1.22.0, che è la build CUDA 12.
!pip uninstall -q -y onnxruntime onnxruntime-gpu
!pip install -q onnx onnxruntime-gpu==1.22.0
import torch, onnxruntime as ort
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())
print('onnxruntime', ort.__version__, '| providers:', ort.get_available_providers())
assert 'CUDAExecutionProvider' in ort.get_available_providers(), \
    'CUDA EP mancante: Runtime -> Riavvia sessione, poi riesegui questa cella'

In [ ]:
# 2) Google Drive: checkpoint + immagini di test
from google.colab import drive
drive.mount('/content/drive')

# === ADATTA QUESTI DUE PATH ===
CHECKPOINT = '/content/drive/MyDrive/PATH/AL/TUO/checkpoint_auc=0.902.pth'
TEST_IMAGES_DIR = '/content/drive/MyDrive/PATH/ALLE/immagini_test'   # cartella con .bmp/.png
IMG_EXTENSION = '.png'  # estensione delle tue immagini di test

import os
assert os.path.isfile(CHECKPOINT), f'checkpoint non trovato: {CHECKPOINT}'
assert os.path.isdir(TEST_IMAGES_DIR), f'cartella immagini non trovata: {TEST_IMAGES_DIR}'
print('ok: checkpoint e immagini trovati')

In [ ]:
# 3) Export ONNX con i PESI VERI (niente --self_test!)
#    Nell'output controlla la riga:  weights: checkpoint:<nome_file>
#    e che la parity check stampi [PASS].
%cd /content/SK-RD4AD
!python export_onnx_from_checkpoint.py "$CHECKPOINT" /content/exports

In [ ]:
# 4) Ispeziona i metadati embedded nel .onnx
import glob, onnxruntime as ort
onnx_path = glob.glob('/content/exports/*.onnx')[0]
sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
meta = dict(sess.get_modelmeta().custom_metadata_map)
for k in sorted(meta):
    print(f'  {k} = {meta[k]}')
assert meta.get('weights_source', '').startswith('checkpoint:'), 'ATTENZIONE: pesi non da checkpoint!'
print('\nOK: il grafo contiene i pesi del checkpoint:', onnx_path)

In [ ]:
# 5) Inferenza completa con inference_simulation su GPU (CUDA EP)
#    Blur/score-source/crop dinamico si auto-configurano dai metadati del modello.
#    Senza --threshold usa un fallback non calibrato (con warning): per il verdetto
#    vero serve la soglia ricalcolata in modo coerente col crop (vedi cella 7).
%cd /content/inference_simulation
!python main.py --model "$onnx_path" \
    --input_dir "$TEST_IMAGES_DIR" --extension "$IMG_EXTENSION" \
    --output_dir /content/results --device cuda

In [ ]:
# 6) Ispezione: score per immagine + heatmap inline
import json, glob
from IPython.display import Image as IPyImage, display

results = sorted(glob.glob('/content/results/*.json'))
for r in results:
    print(r)
with open(results[-1]) as f:
    data = json.load(f)
print(json.dumps({k: v for k, v in data.items() if k != 'batch_throughput_comparison'}, indent=2))

# Cosa aspettarsi con i pesi VERI: raw_anomaly_score che VARIA tra le immagini
# (buone < difettose). Se sono tutti ~identici, qualcosa è ancora storto.
heatmaps = sorted(glob.glob('/content/results/heatmaps/*.png'))[:6]
for h in heatmaps:
    print(h.split('/')[-1])
    display(IPyImage(h, width=320))

In [ ]:
# 7) Calibrazione della soglia + embed nel modello (una volta sola per modello).
#    Calcola gli score sulle immagini BUONE di validazione passando dalla STESSA
#    pipeline del runtime (crop dinamico incluso) e scrive 'calibrated_threshold'
#    nei metadati del .onnx: da quel momento la cella 5 la usa in automatico
#    (niente più fallback col warning) e attiva la visualizzazione pulita
#    threshold-centrica stile eval.py.
GOOD_IMAGES_DIR = '/content/drive/MyDrive/PATH/ALLE/immagini_buone_validazione'
DEFECT_IMAGES_DIR = None  # opzionale: cartella di difetti noti per misurare la separazione

%cd /content/inference_simulation
cmd = f'python calibrate_threshold.py --model "{onnx_path}" --good_dir "{GOOD_IMAGES_DIR}" ' \
      f'--extension "{IMG_EXTENSION}" --device cuda --embed'
if DEFECT_IMAGES_DIR:
    cmd += f' --defect_dir "{DEFECT_IMAGES_DIR}"'
!{cmd}

# Poi RILANCIA la cella 5: userà la soglia embedded e le heatmap threshold-centriche.